# Initialization and Dataset

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

if Path("open_ragbench").exists():
    print("All set!")
else:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="vectara/open_ragbench",
        repo_type="dataset",
        local_dir="./open_ragbench"
    )
    print("Dataset ready!")

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

All set!


In [2]:
# Automatically finds all .json files in your directory
corpus_folder = Path("open_ragbench/pdf/arxiv/corpus")
all_json_files = list(corpus_folder.glob("*.json"))
# all_json_files = all_json_files[:50]

print(f"Found {len(all_json_files)} papers ready for indexing!")

Found 1000 papers ready for indexing!


# Indexing

In [3]:
from haystack import Document, Pipeline
from haystack.components.writers import DocumentWriter
from haystack.components.preprocessors import DocumentSplitter

In [4]:
from docling.chunking import HybridChunker
from haystack_integrations.components.converters.docling import DoclingConverter, ExportType
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore
from haystack_integrations.components.embedders.vllm import VLLMDocumentEmbedder

/home/jack/Documents/University/Internship/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
EXPORT_TYPE = ExportType.DOC_CHUNKS
EMBED_MODEL_ID = "google/embeddinggemma-300m"
# EMBED_MODEL_ID = "mlx-community/Qwen3-Embedding-0.6B-8bit"
# GENERATION_MODEL_ID = "cyankiwi/Qwen3.5-4B-AWQ-4bit"
GENERATION_MODEL_ID = "Qwen/Qwen3-0.6B"

In [6]:
sources = []
for file in all_json_files:
    with open(file) as f:
        paper = json.load(f)

    for section in paper["sections"]:
        sources.append(
            Document(
                content=section["text"][:1000],
                meta={
                    "title": paper["title"],
                    "authors": paper["authors"],
                    "abstract": paper["abstract"],
                    "paper_id": paper["id"],
                    "section_id": section["section_id"],
                    "categories": paper["categories"],
                    "updated": paper["updated"],
                    "published": paper["published"]
                }
            )
        )

In [ ]:
sources[0]

In [7]:
indexing_pipeline = Pipeline()

document_store = QdrantDocumentStore(":memory:", embedding_dim=768) # 768 for gemma, 1024 credo for Qwen

In [8]:
splitter = DocumentSplitter(
    split_length=50,      # ~tokens or words depending on backend
    split_overlap=5
)

converter = DoclingConverter(
    export_type=EXPORT_TYPE, 
    chunker=HybridChunker(tokenizer=EMBED_MODEL_ID)
    )

indexing_embedder = VLLMDocumentEmbedder(model=EMBED_MODEL_ID,
                                         api_base_url="http://localhost:8001/v1")

writer = DocumentWriter(document_store=document_store)

In [9]:
save = splitter.run(sources)

In [10]:
len(save["documents"])

59367

In [11]:
len(sources)

18840

In [12]:
# indexing_pipeline.add_component("converter", converter)
indexing_pipeline.add_component("splitter", splitter)
indexing_pipeline.add_component("embedder", indexing_embedder)
# indexing_pipeline.connect("converter", "embedder")
indexing_pipeline.connect("splitter", "embedder")
indexing_pipeline.add_component("writer", writer)
indexing_pipeline.connect("embedder", "writer")

🚅 Components
  - splitter: DocumentSplitter
  - embedder: VLLMDocumentEmbedder
  - writer: DocumentWriter
🛤️ Connections
  - splitter.documents -> embedder.documents (list[Document])
  - embedder.documents -> writer.documents (list[Document])

In [13]:
indexing_pipeline.run(
    {
        # "converter": {"sources": sources},
        # "embedder": {"sources": sources},
        "splitter": {"documents": sources},
    })

Calculating embeddings: 1856it [02:45, 11.23it/s]
 34%|███▎      | 20000/59367 [00:10<00:19, 2056.38it/s]/home/jack/Documents/University/Internship/.venv/lib/python3.12/site-packages/haystack_integrations/document_stores/qdrant/document_store.py:416: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20100 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self._client.upsert(
59400it [00:30, 1936.39it/s]                           


{'embedder': {'meta': {'model': 'google/embeddinggemma-300m',
   'usage': {'prompt_tokens': 4940644,
    'total_tokens': 4940644,
    'completion_tokens': 0,
    'prompt_tokens_details': None}}},
 'writer': {'documents_written': 59367}}

In [ ]:
docs = document_store.filter_documents()

print(len(docs))

print(docs[0].embedding is not None)

print(type(docs[0].embedding))

print(len(docs[0].embedding) if docs[0].embedding else None)

print(docs[0].embedding[:10] if docs[0].embedding else None)

# RAG

In [14]:
from haystack.components.builders.chat_prompt_builder import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack.components.builders import AnswerBuilder

In [15]:
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

In [16]:
from haystack_integrations.components.embedders.vllm import VLLMTextEmbedder
from haystack_integrations.components.generators.vllm import VLLMChatGenerator

In [17]:
with open("open_ragbench/pdf/arxiv/queries.json") as f:
    queries = json.load(f)

all_questions = [queries[query]["query"] for query in queries]
# all_questions

with open("open_ragbench/pdf/arxiv/answers.json") as f:
    answers = json.load(f)

all_ground_truth_answers = [answers[answer] for answer in answers]
# all_ground_truth_answers

In [18]:
chat_template = [
    ChatMessage.from_system(
        """
        You are a helpful AI assistant.
        Use the provided documents to answer the question.
        If the answer is not in the documents, say so.
        """
    ),
    ChatMessage.from_user(
        """
        Documents:
        {% for doc in documents %}
            {{ doc.content }}
        {% endfor %}

        Question:
        {{query}}
        """
    )
]

rag_pipeline = Pipeline()

In [19]:
rag_embedder = VLLMTextEmbedder(model=EMBED_MODEL_ID,
                                api_base_url="http://localhost:8001/v1")

retriever = QdrantEmbeddingRetriever(document_store=document_store, top_k=5)

prompt_builder = ChatPromptBuilder(template=chat_template, required_variables={"query", "documents"})

llm = VLLMChatGenerator(model=GENERATION_MODEL_ID,
                        api_base_url="http://localhost:8000/v1")

answer_builder = AnswerBuilder()

In [20]:
rag_pipeline.add_component("embedder", rag_embedder)
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.connect("embedder.embedding", "retriever")
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.connect("retriever", "prompt_builder")
rag_pipeline.add_component("llm", llm)
rag_pipeline.connect("prompt_builder", "llm")
rag_pipeline.add_component("answer_builder", answer_builder)
rag_pipeline.connect("llm", "answer_builder")
rag_pipeline.connect("retriever", "answer_builder")

🚅 Components
  - embedder: VLLMTextEmbedder
  - retriever: QdrantEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - llm: VLLMChatGenerator
  - answer_builder: AnswerBuilder
🛤️ Connections
  - embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - retriever.documents -> answer_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])
  - llm.replies -> answer_builder.replies (list[ChatMessage])

In [ ]:
rag_pipeline.show()

In [21]:
response = rag_pipeline.run(
    {
        "embedder": {"text": all_questions[0]},
        "prompt_builder": {"query": all_questions[0]},
        "answer_builder": {"query": all_questions[0]},
    })

In [22]:
response["answer_builder"]["answers"][0].data

'<think>\nOkay, let\'s see. The user is asking about the challenges in estimating output impedance in inverter-based grids based on the provided documents.\n\nFirst, I need to go through the documents to find relevant information. The abstract mentions non-invasive estimation using local measurements from inverters. The introduction talks about distributed energy resources and the importance of output line impedance. Then there\'s a section on grid modeling using Thevenin\'s theorem, which relates to impedance estimation. \n\nLooking at the grid modeling part, it says the grid is represented as a voltage source and impedance, and that output impedance is defined between the inverter and grid. The document also mentions that the grid represents an abstinent part, which might be a translation issue. \n\nWait, in the document, there\'s a part that says "the grid, in this context, represents an abst" which might be a typo. But regardless, the key points are about the use of Thevenin\'s the